# NYC Taxi Trip Duration — Kompletan ML Pipeline

**Zadatak:** Predvidi trajanje taksi vožnje u sekundama.

Ovaj notebook prati sve korake iz `prakticni_primer.md`.
Pokreni svaku ćeliju redom (Shift+Enter).


## Korak 1: Uvoz biblioteka


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import math
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

print('Biblioteke ucitane.')


## Korak 2: Učitavanje podataka


In [ ]:
train = pd.read_csv('Train.csv')
test  = pd.read_csv('Test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
train.head()


## Korak 3: Istraživanje podataka (EDA)


In [ ]:
print(train.info())
print('---')
print(train.describe())


In [ ]:
print('Nedostajuce vrednosti u train:')
print(train.isnull().sum())


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(train['trip_duration'], bins=50, color='steelblue')
plt.title('Distribucija trajanja voznje')
plt.xlabel('Sekunde')

plt.subplot(1, 2, 2)
plt.hist(np.log1p(train['trip_duration']), bins=50, color='coral')
plt.title('Log distribucija')
plt.xlabel('log(Sekunde + 1)')

plt.tight_layout()
plt.show()


## Korak 4: Feature Engineering

### 4a. Log-transformacija targeta
Koristimo `log1p` jer RMSLE metrika radi u log prostoru.


In [ ]:
train['log_trip_duration'] = np.log1p(train['trip_duration'])


### 4b. Obrada datuma


In [ ]:
train['pickup_datetime'] = pd.to_datetime(train['pickup_datetime'])
test['pickup_datetime']  = pd.to_datetime(test['pickup_datetime'])

for df in [train, test]:
    df['sat']          = df['pickup_datetime'].dt.hour
    df['dan_sedmice']  = df['pickup_datetime'].dt.dayofweek
    df['mesec']        = df['pickup_datetime'].dt.month
    df['dan_u_mesecu'] = df['pickup_datetime'].dt.day

    df['sat_sin'] = np.sin(2 * math.pi * df['sat'] / 24)
    df['sat_cos'] = np.cos(2 * math.pi * df['sat'] / 24)

print('Datum feature-i kreirani.')


### 4c. Euklidska distanca — Najvažniji feature


In [ ]:
for df in [train, test]:
    df['distanca'] = np.sqrt(
        (df['pickup_latitude']  - df['dropoff_latitude'])**2 +
        (df['pickup_longitude'] - df['dropoff_longitude'])**2
    )

print('Distanca izracunata.')
print(train['distanca'].describe())


### 4d. Enkodiranje kategorijskih kolona


In [ ]:
for df in [train, test]:
    df['vendor_id'] = df['vendor_id'].map({'CMT': 0, 'VTS': 1})
    df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'N': 0, 'Y': 1})

print('Enkodiranje zavrseno.')


## Korak 5: Rukovanje NaN vrednostima


In [ ]:
median_passengers = train['passenger_count'].median()

train['passenger_count'] = train['passenger_count'].fillna(median_passengers)
test['passenger_count']  = test['passenger_count'].fillna(median_passengers)

print('NaN popunjeni. Provera:')
print(train.isnull().sum())


## Korak 6: Priprema feature-a i Train/Validation split


In [ ]:
FEATURES = [
    'vendor_id',
    'passenger_count',
    'pickup_longitude', 'pickup_latitude',
    'dropoff_longitude', 'dropoff_latitude',
    'store_and_fwd_flag',
    'sat', 'dan_sedmice', 'mesec', 'dan_u_mesecu',
    'sat_sin', 'sat_cos',
    'distanca'
]

TARGET = 'log_trip_duration'

X = train[FEATURES]
y = train[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train size:', X_train.shape)
print('Val size:  ', X_val.shape)


## Korak 7: Trening XGBoost modela


In [ ]:
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print('Trening zavrsen.')


## Korak 8: Evaluacija na validation setu


In [ ]:
y_pred_val = model.predict(X_val)

rmsle = np.sqrt(mean_squared_log_error(
    np.expm1(y_val),
    np.expm1(y_pred_val)
))

print(f'Validation RMSLE: {rmsle:.4f}')


## Korak 9: Podešavanje hiperparametara

Pokušaj promijeniti parametre i vidjeti da li se RMSLE poboljšava.
Ponavljaj Korak 7 i 8 dok ne dobiješ što manji RMSLE.


In [ ]:
model_v2 = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

model_v2.fit(X_train, y_train)
y_pred_v2 = model_v2.predict(X_val)

rmsle_v2 = np.sqrt(mean_squared_log_error(
    np.expm1(y_val),
    np.expm1(y_pred_v2)
))

print(f'V1 RMSLE: {rmsle:.4f}')
print(f'V2 RMSLE: {rmsle_v2:.4f}')


## Korak 10: Re-train na celom train setu

Kad nađeš najbolje parametre, treniraj na **svim** podacima.


In [ ]:
# Uzmi parametre koji su dali najbolji RMSLE u koraku 9
best_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X, y)
print('Finalni model istreniran na celom train setu.')


## Korak 11: Predikcija na test setu


In [ ]:
X_test = test[FEATURES]

log_predictions = best_model.predict(X_test)
predictions = np.expm1(log_predictions)

print('Prve 5 predikcija (sekunde):')
print(predictions[:5])


## Korak 12: Generisanje submission fajla


In [ ]:
submission = pd.DataFrame({
    'id':            test['id'],
    'trip_duration': predictions
})

submission.to_csv('submission.csv', index=False)
print('submission.csv sacuvan!')
submission.head(10)


## ✅ Gotovo!

Submission fajl je spreman. Upload-uj `submission.csv` na platformu.

**Kako dalje poboljšati score:**
- Dodaj nove feature-e (npr. Manhattan distanca, rushour flag...)
- Probaj više vrijednosti hiperparametara
- Probaj ukloniti outlier-e iz trening seta
